# pycograd — GRU & LSTM, gated RNNs in the ambient-weights DSL

The two classic gated recurrent cells — the **GRU** and the **LSTM** — are the
workhorses of pre-transformer sequence modeling. Both are tiny: a couple of
`sigmoid` gates around a `tanh` candidate, applied one timestep at a time.

This notebook builds a **character-level GRU language model** with pycograd's
ambient-weights DSL: declare the parameters once with `params{...}`, write the
forward once with the weights referred to by **bare name**, and the *same* code
trains on the numpy tape, batches under `vmap`, and compiles to **torch / jax /
tf**. An LSTM in the same style follows at the end.

Every layer is ordinary numpy (`@`, `np.tanh`, the `sigmoid` helper,
`np.concatenate`), so pycograd differentiates it by instrumenting on demand — no
hand-written gradients. The reusable cells (`gru_cell`, `lstm_cell`, the `*_scan`
sequence loops) also ship in `pycograd.examples.models`.

In [1]:
%load_ext pycograd

## A tiny corpus

Character-level next-character prediction. The corpus is deliberately small so a
tiny model memorizes it quickly.

In [2]:
import numpy as np
from pycograd import vmap
from pycograd.examples.models import softmax_last

text = "pycograd makes gru tiny. "
vocab = sorted(set(text))
stoi = {c: i for i, c in enumerate(vocab)}
ids = np.array([stoi[c] for c in text])
onehot = np.eye(len(vocab))[ids]            # (T, V) one-hot rows
V = len(vocab)
print("text  :", repr(text))
print("vocab :", "".join(vocab), " (V=%d, T=%d)" % (V, len(ids)))

# inputs predict the *next* char: feed prefix 0..T-2, target chars 1..T-1
X_oh, Y_oh, Y_ids = onehot[:-1], onehot[1:], ids[1:]

text  : 'pycograd makes gru tiny. '
vocab :  .acdegikmnoprstuy  (V=18, T=25)


## Helpers

`sigmoid` is the gate activation; `next_char_ce` is the mean next-character
cross-entropy. Both are plain numpy — pycograd differentiates them transparently.

In [3]:
def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

def next_char_ce(logits, onehot):          # mean over positions
    return -np.mean(np.sum(onehot * np.log(softmax_last(logits) + 1e-12), axis=-1))

## The model — a GRU, written once

A GRU updates a hidden state `h` one timestep at a time. Each step computes an
**update gate** `z` and a **reset gate** `r`, forms a candidate `n` (the reset
gate decides how much of the old state feeds the candidate), and blends:

$$h_t = (1 - z)\,n + z\,h_{t-1}.$$

The embedding is `onehot @ emb` — an embedding lookup written as a matmul, which
keeps the net free of fancy indexing. We carry the state as a `(1, D)` **row** and
slice each timestep as `x[t : t+1]`, so every matmul is rank-2 — TensorFlow's `@`
needs rank ≥ 2, and the rows lower cleanly to all three backends. The hidden seed
`np.expand_dims(bz * 0.0, 0)` takes its shape from a bias param (**not** a
hardcoded constant), so the same scan still vectorizes under `vmap` (each sequence
gets its own logical shape).

We write `gru` once, as a top-level function referring to the weights by bare
name (`emb`, `Wz`, …). `with params{...} as weights:` injects a late-bound proxy
for each name, so the *same* `gru` is bound to tape nodes by `weights.grad`, to a
batch by `vmap`, and to framework tensors by the compiler — no rewrite per use.

In [4]:
def gru(oh):                                       # oh: (T, V) one-hot rows
    x = oh @ emb                                   # (T, D) embedding (a matmul)
    h = np.expand_dims(bz * 0.0, 0)                # (1, D) zero hidden row
    outs = []
    for t in range(x.shape[0]):                    # scan over time
        xt = x[t : t + 1]                          # (1, D) row -> rank-2 matmuls
        z = sigmoid(xt @ Wz + h @ Uz + bz)         # update gate
        r = sigmoid(xt @ Wr + h @ Ur + br)         # reset gate
        n = np.tanh(xt @ Wn + (r * h) @ Un + bn)   # candidate
        h = (1 - z) * n + z * h
        outs.append(h)
    H = np.concatenate(outs, axis=0)               # (T, D) hidden states
    return H @ head_w + head_b                      # (T, V) next-char logits

def objective():
    return next_char_ce(gru(X_oh), Y_oh)

Now declare the weights with `params{...}` and train. The names below are
exactly the bare names `gru` refers to; `weights.grad(objective)` binds them to
the numpy tape and returns gradients, and `weights.step` updates them in place.

In [5]:
rng = np.random.default_rng(0)
D = 24                                       # model width
s = 0.1

with params{
    emb    = s * rng.standard_normal((V, D))
    Wz = s*rng.standard_normal((D, D)); Uz = s*rng.standard_normal((D, D)); bz = np.zeros(D)
    Wr = s*rng.standard_normal((D, D)); Ur = s*rng.standard_normal((D, D)); br = np.zeros(D)
    Wn = s*rng.standard_normal((D, D)); Un = s*rng.standard_normal((D, D)); bn = np.zeros(D)
    head_w = s*rng.standard_normal((D, V)); head_b = np.zeros(V)
} as weights:
    first = last = None
    for _ in range(400):
        value, grads = weights.grad(objective)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, 0.5)
    logits = gru(X_oh)

print("loss %.4f -> %.4f  (400 SGD steps)" % (first, last))
print("train accuracy:", float(np.mean(np.argmax(np.asarray(logits), axis=-1) == Y_ids)))

loss 2.8888 -> 0.0249  (400 SGD steps)
train accuracy: 1.0


## Batching with `vmap`

`gru` was written for one `(T, V)` sequence. `vmap` vectorizes it over a batch of
sequences in a single pass — no Python loop over the batch — and the result
matches the loop oracle exactly. The per-timestep recurrence batches for free
because the `(1, D)` hidden seed carries each sequence's own logical shape.

In [6]:
# a batch of (T, V) one-hot sequences (random tokens, just to exercise batching)
B, T = 5, len(X_oh)
batch = np.eye(V)[np.random.default_rng(1).integers(0, V, size=(B, T))]

with weights:
    batched = np.asarray(vmap(gru)(batch))                 # one vectorized pass
    looped  = np.stack([np.asarray(gru(batch[i])) for i in range(B)])  # oracle

print("vmap output shape:", batched.shape)
print("matches the python loop:", np.allclose(batched, looped, atol=1e-9))

vmap output shape: (5, 24, 18)
matches the python loop: True


## Compiling to torch / jax / tf

`weights.grad(objective, backend=...)` runs the *same* objective through a
framework's own autodiff instead of pycograd's numpy tape. We check the framework
gradients against the numpy tape (which is finite-difference-checked in the test
suite), so agreement validates the whole compile path. `jit=True` traces and
compiles the gradient **once** (jax.jit / tf.function / torch make_fx), not per
step.

The per-timestep `(1, D)` rows keep every matmul rank-2, so the scan lowers to all
three frameworks — TensorFlow included (its `@` needs rank ≥ 2).

In [7]:
with weights:
    v_np, g_np = weights.grad(objective)                    # pycograd's numpy tape
    for backend in ("torch", "jax", "tf"):
        v_be, g_be = weights.grad(objective, backend=backend, jit=True)
        worst = max(np.max(np.abs(np.asarray(g_be[k]) - np.asarray(g_np[k]))) for k in weights)
        print("%-5s  value=%.6f  max|grad - grad_numpy| = %.1e" % (backend, float(v_be), worst))
    print("numpy value=%.6f" % float(v_np))

torch  value=0.024725  max|grad - grad_numpy| = 3.3e-18


jax    value=0.024725  max|grad - grad_numpy| = 4.3e-18


tf     value=0.024725  max|grad - grad_numpy| = 2.7e-18
numpy value=0.024725


## Generating text — the recurrent form

A GRU is *already* recurrent: the training scan above carries `h` forward one
token at a time, so sampling needs no separate "parallel vs recurrent" rewrite —
we just run the same step and feed each sampled character back in. We read the
trained weights straight out of the `ParamDict`.

In [8]:
W = {k: np.asarray(weights[k].value) for k in weights}   # trained arrays

def generate(prime, n_new, temp=0.5, seed=0):
    g = np.random.default_rng(seed)
    h = np.zeros(D)                                  # the GRU's recurrent state
    out = list(prime)

    def step(token):
        nonlocal h
        x = W["emb"][token]
        z = sigmoid(x @ W["Wz"] + h @ W["Uz"] + W["bz"])
        r = sigmoid(x @ W["Wr"] + h @ W["Ur"] + W["br"])
        n = np.tanh(x @ W["Wn"] + (r * h) @ W["Un"] + W["bn"])
        h = (1 - z) * n + z * h
        return h @ W["head_w"] + W["head_b"]

    logits = None
    for c in prime:                                  # warm the state on the prime
        logits = step(stoi[c])
    for _ in range(n_new):
        p = np.exp((logits - logits.max()) / temp); p /= p.sum()
        nxt = int(g.choice(V, p=p))
        out.append(vocab[nxt])
        logits = step(nxt)
    return "".join(out)

print(repr(generate("pyco", 21)))

'pycograd makes gru tiny. '


## An LSTM in the same style

The LSTM adds a **cell state** `c` alongside the hidden `h`, with input / forget /
output gates (`i` / `f` / `o`) around a `tanh` candidate `g`:

$$c_t = f\,c_{t-1} + i\,g, \qquad h_t = o\,\tanh(c_t).$$

Same DSL: write the forward once with bare names, declare the weights, train. The
`bf` forget-gate bias seeds both zero states (`bf * 0.0`).

In [9]:
def lstm(oh):
    x = oh @ Lemb                                   # (T, D) embedding
    h = np.expand_dims(Lbf * 0.0, 0)                # (1, D) zero hidden row
    c = np.expand_dims(Lbf * 0.0, 0)                # (1, D) zero cell row
    outs = []
    for t in range(x.shape[0]):
        xt = x[t : t + 1]                           # (1, D) row -> rank-2 matmuls
        i = sigmoid(xt @ Wi + h @ Ui + Lbi)         # input gate
        f = sigmoid(xt @ Wf + h @ Uf + Lbf)         # forget gate
        gg = np.tanh(xt @ Wg + h @ Ug + Lbg)        # candidate cell
        o = sigmoid(xt @ Wo + h @ Uo + Lbo)         # output gate
        c = f * c + i * gg
        h = o * np.tanh(c)
        outs.append(h)
    return np.concatenate(outs, axis=0) @ Lhead_w + Lhead_b

def lstm_objective():
    return next_char_ce(lstm(X_oh), Y_oh)

rng = np.random.default_rng(0)
def Wd():  return s * rng.standard_normal((D, D))

with params{
    Lemb = s * rng.standard_normal((V, D))
    Wi = Wd(); Ui = Wd(); Lbi = np.zeros(D)
    Wf = Wd(); Uf = Wd(); Lbf = np.zeros(D)
    Wg = Wd(); Ug = Wd(); Lbg = np.zeros(D)
    Wo = Wd(); Uo = Wd(); Lbo = np.zeros(D)
    Lhead_w = s * rng.standard_normal((D, V)); Lhead_b = np.zeros(V)
} as lstm_weights:
    first = last = None
    for _ in range(500):
        value, grads = lstm_weights.grad(lstm_objective)
        first = float(value) if first is None else first
        last = float(value)
        lstm_weights.step(grads, 0.5)
    logits = lstm(X_oh)

print("LSTM loss %.4f -> %.4f  (500 SGD steps)" % (first, last))
print("train accuracy:", float(np.mean(np.argmax(np.asarray(logits), axis=-1) == Y_ids)))

LSTM loss 2.8912 -> 0.0592  (500 SGD steps)
train accuracy: 1.0


## More

- `pycograd_rwkv_demo.ipynb` — a recurrent net that trains in parallel and runs
  in an O(1)-per-token form.
- `pycograd_demo.ipynb` — the DSL from scratch (MLP, highway nets, a Transformer block).
- `pycograd_vmap_demo.ipynb` — batching with `vmap`.
- `pycograd_compile_{torch,jax,tf}_demo.ipynb` — compiling the DSL to each framework.

The gated cells (`rnn_cell`, `gru_cell`, `lstm_cell`) and their sequence scans
(`rnn_scan`, `gru_scan`, `lstm_scan`) live in `pycograd.examples.models`.